# 01 — Exploration du jeu de données Mobilisator

Ce notebook charge les données électorales et dresse un premier portrait statistique des 9 989 communes françaises incluses dans le dataset.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
DATA_FILE = Path('../../public/cities/cities-data.json')

In [ ]:
with open(DATA_FILE) as f:
    raw = json.load(f)

cities = list(raw.values())
print(f"{len(cities)} communes chargées")
print("\nStructure d'une commune :")
print(json.dumps({k: v for k, v in cities[0].items() if k not in ('Tour 1', 'Tour 2', 'population', 'Analyse')}, indent=2))

In [ ]:
rows = []
for c in cities:
    t1 = c.get('Tour 1', {})
    t2 = c.get('Tour 2')
    analyse = c.get('Analyse', {})
    pop = c.get('population', {})

    row = {
        'id': c['id'],
        'nom': c['nom_standard'],
        'slug': c['slug'],
        'code_dept': c['code_departement'],
        'dept': c['libelle_departement'],
        # Tour 1
        't1_inscrits': t1.get('Inscrits'),
        't1_abstentions': t1.get('Abstentions'),
        't1_pct_abs': t1.get('% Abs/Ins'),
        't1_votants': t1.get('Votants'),
        't1_exprimes': t1.get('Exprimés'),
        't1_blancs': t1.get('Blancs'),
        't1_nuls': t1.get('Nuls'),
        't1_nb_listes': len(t1.get('resultats', [])),
        # Tour 2
        'a_tour2': t2 is not None,
        't2_pct_abs': t2.get('% Abs/Ins') if t2 else None,
        # Analyse
        'votes_decisifs': analyse.get('Votes décisifs'),
        'tour_decisif': analyse.get('tour décisif'),
        'majeurs': analyse.get('majeurs'),
        'non_votants_1839': analyse.get('Non votants de 18-39'),
        'pop_1839': analyse.get('Pop 18-39'),
        'pop_18plus': analyse.get('Pop 18+'),
        'non_votants': analyse.get('Non votants'),
        'part_ne_votant_pas': analyse.get('Part ne votant pas'),
    }

    # Population totale
    if pop:
        row['pop_totale'] = sum(pop.values())
    else:
        row['pop_totale'] = None

    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
df.head(3)

## Statistiques descriptives

In [ ]:
df[['t1_inscrits', 't1_pct_abs', 't1_nb_listes', 'pop_totale', 'part_ne_votant_pas']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution du taux d'abstention T1
axes[0].hist(df['t1_pct_abs'].dropna(), bins=50, edgecolor='white', color='steelblue')
axes[0].set_title("Distribution de l'abstention (Tour 1)")
axes[0].set_xlabel("% Abstention")
axes[0].set_ylabel("Nombre de communes")

# Distribution de la population
df_pop = df['pop_totale'].dropna()
axes[1].hist(df_pop[df_pop < 50000], bins=60, edgecolor='white', color='coral')
axes[1].set_title("Population des communes (< 50 000 hab.)")
axes[1].set_xlabel("Population")

# Nombre de listes T1
df['t1_nb_listes'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title("Nombre de listes au Tour 1")
axes[2].set_xlabel("Nb listes")
axes[2].set_ylabel("Nb communes")

plt.tight_layout()
plt.savefig('../outputs/01_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f"Communes avec Tour 2 : {df['a_tour2'].sum()} ({df['a_tour2'].mean()*100:.1f}%)")
print(f"Communes sans données population : {df['pop_totale'].isna().sum()}")
print(f"\nTop 10 communes par nombre d'inscrits :")
df.nlargest(10, 't1_inscrits')[['nom', 'dept', 't1_inscrits', 't1_pct_abs']].to_string(index=False)

## Classification démographique

Estimation de l'**âge médian** de chaque commune à partir de la pyramide des âges (tranches INSEE), puis clustering **K-Means** sur deux dimensions : âge médian estimé et log(population). L'objectif est de dégager des profils-types de communes.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# --- Estimation de l'âge médian par interpolation linéaire dans les tranches ---
BAND_STARTS = {'0-2': 0, '3-5': 3, '6-10': 6, '11-17': 11, '18-24': 18,
               '25-39': 25, '40-54': 40, '55-64': 55, '65-79': 65, '80+': 80}
BAND_WIDTHS  = {'0-2': 3, '3-5': 3, '6-10': 5, '11-17': 7, '18-24': 7,
               '25-39': 15, '40-54': 15, '55-64': 10, '65-79': 15, '80+': 20}
BANDS_ORDER  = list(BAND_STARTS.keys())

def median_age_from_pop(pop: dict) -> float | None:
    if not pop:
        return None
    counts = {}
    for key, n in pop.items():
        band = key[1:]  # supprime préfixe F/H
        counts[band] = counts.get(band, 0) + n
    total = sum(counts.values())
    if total == 0:
        return None
    half = total / 2
    cumulative = 0.0
    for band in BANDS_ORDER:
        n = counts.get(band, 0)
        if cumulative + n >= half:
            frac = (half - cumulative) / n if n > 0 else 0.5
            return BAND_STARTS[band] + frac * BAND_WIDTHS[band]
        cumulative += n
    return 90.0

df['age_median_est'] = [median_age_from_pop(c.get('population')) for c in cities]
df['log_pop'] = np.log10(df['pop_totale'].clip(lower=1))

valid = df[['age_median_est', 'log_pop']].dropna()
print(f"Communes avec données complètes : {len(valid)} / {len(df)}")
print(f"Âge médian moyen : {df['age_median_est'].mean():.1f} ans")
print(f"Plage : {df['age_median_est'].min():.1f} – {df['age_median_est'].max():.1f} ans")

In [ ]:
# --- K-Means : méthode du coude puis classification finale ---
features = df[['age_median_est', 'log_pop']].dropna()
scaler = StandardScaler()
X = scaler.fit_transform(features)

# Méthode du coude
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(K_range), inertias, 'o-', color='steelblue', linewidth=2)
ax.set_xlabel("Nombre de clusters (k)")
ax.set_ylabel("Inertie (within-cluster sum of squares)")
ax.set_title("Méthode du coude — choix de k")
ax.axvline(6, color='coral', linestyle='--', label='k=6 retenu')
ax.legend()
plt.tight_layout()
plt.show()

# Classification finale k=6
K = 6
km_final = KMeans(n_clusters=K, random_state=42, n_init=10)
df.loc[features.index, 'cluster'] = km_final.fit_predict(X)
df['cluster'] = df['cluster'].astype('Int64')
print(df['cluster'].value_counts().sort_index().rename("nb communes"))

In [ ]:
# --- Nuage de points coloré par cluster ---
palette = sns.color_palette('tab10', K)

fig, ax = plt.subplots(figsize=(13, 7))
for c_id in sorted(df['cluster'].dropna().unique()):
    sub = df[df['cluster'] == c_id]
    ax.scatter(
        sub['age_median_est'], sub['pop_totale'],
        c=[palette[int(c_id)]], label=f"Cluster {int(c_id) + 1}",
        s=18, alpha=0.55, edgecolors='none'
    )

ax.set_yscale('log')
ax.set_xlabel("Âge médian estimé (ans)", fontsize=12)
ax.set_ylabel("Population totale (échelle log)", fontsize=12)
ax.set_title("Classification démographique des communes — K-Means (k=6)", fontsize=14)
ax.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc='upper left')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', ' ')))
plt.tight_layout()
plt.savefig('../outputs/01_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Caractérisation des clusters ---
summary = df.groupby('cluster').agg(
    nb_communes   = ('nom', 'count'),
    pop_mediane   = ('pop_totale', 'median'),
    pop_moy       = ('pop_totale', 'mean'),
    age_median    = ('age_median_est', 'mean'),
    abs_moy       = ('t1_pct_abs', 'mean'),
).round(1)

# Reconstruit les centroïdes dans l'espace original
centroids_orig = scaler.inverse_transform(km_final.cluster_centers_)  # [age, log_pop]

def describe_cluster(c_id: int) -> str:
    age   = centroids_orig[c_id][0]
    pop   = 10 ** centroids_orig[c_id][1]
    abs_r = summary.loc[c_id, 'abs_moy']
    n     = int(summary.loc[c_id, 'nb_communes'])

    if pop > 150_000:
        size = "Grandes métropoles"
    elif pop > 30_000:
        size = "Villes moyennes"
    elif pop > 6_000:
        size = "Petites villes"
    elif pop > 1_500:
        size = "Bourgs"
    else:
        size = "Villages"

    if age < 36:
        age_desc = "population très jeune"
    elif age < 41:
        age_desc = "population jeune"
    elif age < 44:
        age_desc = "population d'âge intermédiaire"
    elif age < 48:
        age_desc = "population vieillissante"
    else:
        age_desc = "population très âgée"

    abs_desc = "abstention élevée" if abs_r > 40 else ("abstention modérée" if abs_r > 33 else "abstention faible")

    return (f"{size} — {age_desc} (âge médian {age:.1f} ans), "
            f"{abs_desc} ({abs_r:.1f} %), {n} communes.")

print("=" * 72)
print("PROFILS DÉMOGRAPHIQUES DES COMMUNES FRANÇAISES (K-Means, k=6)")
print("=" * 72)

for c_id in sorted(df['cluster'].dropna().unique()):
    c_id = int(c_id)
    desc  = describe_cluster(c_id)
    top5  = (df[df['cluster'] == c_id]
             .nlargest(5, 'pop_totale')[['nom', 'dept', 'pop_totale', 'age_median_est']])
    top5_str = top5.to_string(index=False)
    print(f"\nCluster {c_id + 1} : {desc}")
    print(f"  Exemples (plus grandes communes du cluster) :")
    for _, row in top5.iterrows():
        print(f"    • {row['nom']} ({row['dept']}) — {int(row['pop_totale']):,} hab., âge médian {row['age_median_est']:.1f} ans".replace(',', ' '))
print()